# ANOVA (분산분석) 예제

이 노트북은 Python을 사용하여 일원배치 분산분석(One-way ANOVA)을 수행하는 예제를 보여줍니다.
ANOVA는 세 개 이상의 그룹 간의 평균에 통계적으로 유의미한 차이가 있는지 여부를 검정하는 데 사용됩니다.

**라이브러리:**
- `pandas`: 데이터 조작 및 분석을 위해 사용됩니다.
- `scipy.stats`: 과학 계산 및 통계 분석을 위한 함수를 제공합니다. `f_oneway` 함수를 사용하여 ANOVA를 수행합니다.
- `statsmodels`: 더 상세한 통계 분석 및 모델링을 위한 라이브_러리입니다. OLS(Ordinary Least Squares) 모델을 사용하여 ANOVA 결과를 확인할 수 있습니다.
- `statsmodels.stats.multicomp`: 다중 비교를 위한 Tukey's HSD 검정을 제공합니다.


In [ ]:
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- 1. 샘플 데이터 생성 ---
# 세 가지 다른 그룹(A, B, C)에 대한 데이터를 생성합니다.
# 각 그룹은 특정 처리를 받은 식물의 성장률을 나타낸다고 가정합니다.
data = {'group': ['A', 'A', 'A', 'A', 'A', 
                  'B', 'B', 'B', 'B', 'B', 
                  'C', 'C', 'C', 'C', 'C'],
        'growth': [2.5, 3.0, 2.8, 3.2, 2.9,
                   3.8, 3.5, 4.0, 3.9, 3.6,
                   3.0, 2.9, 3.2, 3.1, 3.3]}
df = pd.DataFrame(data)

print("--- 샘플 데이터 ---")
print(df)
print("\\n" + "="*30 + "\\n")


# --- 2. ANOVA 수행 (Scipy 사용) ---
# 각 그룹의 데이터를 추출합니다.
group_a = df[df['group'] == 'A']['growth']
group_b = df[df['group'] == 'B']['growth']
group_c = df[df['group'] == 'C']['growth']

# 일원배치 분산분석(One-way ANOVA)을 수행합니다.
f_statistic, p_value = stats.f_oneway(group_a, group_b, group_c)

print("--- Scipy를 이용한 ANOVA 결과 ---")
print(f"F-statistic: {f_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

# 결과 해석
alpha = 0.05  # 유의수준
if p_value < alpha:
    print("P-value가 유의수준보다 작으므로, 그룹 간에 통계적으로 유의미한 차이가 있습니다.")
else:
    print("P-value가 유의수준보다 크므로, 그룹 간에 통계적으로 유의미한 차이가 없습니다.")
print("\\n" + "="*30 + "\\n")


# --- 3. ANOVA 수행 (Statsmodels 사용) ---
# Statsmodels를 사용하면 더 상세한 분석 결과를 얻을 수 있습니다.
# OLS(Ordinary Least Squares) 모델을 사용하여 ANOVA를 수행합니다.
model = ols('growth ~ C(group)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("--- Statsmodels를 이용한 ANOVA 결과 ---")
print(anova_table)
print("\\n" + "="*30 + "\\n")


# --- 4. 사후 분석 (Tukey's HSD) ---
# ANOVA 결과가 유의미할 경우, 어떤 그룹들 간에 차이가 있는지 확인하기 위해 사후 분석을 수행합니다.
# Tukey's Honestly Significant Difference (HSD) test를 사용합니다.
tukey_result = pairwise_tukeyhsd(endog=df['growth'], groups=df['group'], alpha=alpha)

print("--- Tukey's HSD 사후 분석 결과 ---")
print(tukey_result)

# 사후 분석 결과 해석
# reject=True는 해당 그룹 쌍 간의 평균에 유의미한 차이가 있음을 의미합니다.


---
# 이원배치 분산분석 (Two-way ANOVA) 및 상호작용 항

이원배치 분산분석은 두 개의 독립 변수(요인)가 하나의 종속 변수에 미치는 영향을 분석할 때 사용됩니다.

**주요 분석 내용:**
1.  **주효과 (Main Effect)**: 각 독립 변수가 다른 변수와 관계없이 종속 변수에 미치는 독립적인 효과.
2.  **상호작용 효과 (Interaction Effect)**: 한 독립 변수의 효과가 다른 독립 변수의 수준에 따라 달라지는 효과.

`statsmodels`에서는 `*` 또는`:` 연산자를 사용하여 상호작용 항을 모델에 추가할 수 있습니다.
- `Y ~ C(A) + C(B)`: A와 B의 주효과만 고려 (상호작용 없음)
- `Y ~ C(A) + C(B) + C(A):C(B)`: A와 B의 주효과 및 상호작용 효과를 모두 고려
- `Y ~ C(A) * C(B)`: 위와 동일한 표현. 주효과와 상호작용 효과를 모두 포함하는 축약형입니다.


In [ ]:
# --- 5. 이원배치 분산분석 (Two-way ANOVA) 예제 ---

# 5-1. 샘플 데이터 생성
# 두 개의 독립 변수(group, fertilizer)를 갖는 데이터를 생성합니다.
# group: 식물의 종류 (A, B)
# fertilizer: 비료의 종류 (X, Y)
data_2way = {'group': ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A',
                      'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B'],
             'fertilizer': ['X', 'X', 'X', 'X', 'Y', 'Y', 'Y', 'Y',
                            'X', 'X', 'X', 'X', 'Y', 'Y', 'Y', 'Y'],
             'growth': [2.5, 2.8, 2.6, 2.7, 3.0, 3.2, 3.1, 3.3,  # A-X, A-Y
                        3.2, 3.4, 3.3, 3.1, 4.0, 4.2, 4.1, 4.3]} # B-X, B-Y
df_2way = pd.DataFrame(data_2way)

print("--- 이원배치 ANOVA 샘플 데이터 ---")
print(df_2way)
print("\\n" + "="*40 + "\\n")


# 5-2. 이원배치 ANOVA 수행 (Statsmodels 사용)
# 'growth ~ C(group) * C(fertilizer)' 포뮬라는 아래와 동일합니다.
# 'growth ~ C(group) + C(fertilizer) + C(group):C(fertilizer)'
# C(group): group 변수의 주효과
# C(fertilizer): fertilizer 변수의 주효과
# C(group):C(fertilizer): 두 변수 간의 상호작용 효과
model_2way = ols('growth ~ C(group) * C(fertilizer)', data=df_2way).fit()
anova_table_2way = sm.stats.anova_lm(model_2way, typ=2)

print("--- Statsmodels를 이용한 이원배치 ANOVA 결과 ---")
print(anova_table_2way)
print("\\n" + "="*40 + "\\n")

# 결과 해석
# - C(group)의 P-value (PR(>F)): 식물 종류(group)에 따른 성장률에 유의미한 차이가 있는지 확인합니다.
# - C(fertilizer)의 P-value (PR(>F)): 비료 종류(fertilizer)에 따른 성장률에 유의미한 차이가 있는지 확인합니다.
# - C(group):C(fertilizer)의 P-value (PR(>F)): 식물 종류와 비료 종류 간에 상호작용 효과가 있는지 확인합니다.
#   - 상호작용 효과의 p-value가 유의수준(e.g., 0.05)보다 작으면, 한 요인의 효과가 다른 요인의 수준에 따라 달라진다고 해석할 수 있습니다.


# 5-3. 사후 분석 (상호작용이 유의미할 경우)
# 상호작용 효과가 통계적으로 유의미하다면, 주효과를 해석하는 것은 의미가 없을 수 있습니다.
# 대신, 모든 조합(A-X, A-Y, B-X, B-Y) 간의 평균을 비교해야 합니다.
df_2way['combination'] = df_2way['group'] + "_" + df_2way['fertilizer']

tukey_2way = pairwise_tukeyhsd(endog=df_2way['growth'], groups=df_2way['combination'], alpha=0.05)

print("--- 이원배치 ANOVA 사후 분석 (Tukey's HSD) ---")
print(tukey_2way)
